# Notebook 2: Everywhere the Rover Can Reach (span)

**Predict, then run.** Run the two setup cells first.

**Setup.** The next cell is the drawing library, `vecdraw`. It is short, and you are allowed to read it and change it.

In [ ]:
%%writefile vecdraw.py
"""
vecdraw - a tiny helper for drawing vectors on graph paper.

READ ME! Nothing in here is magic. Every function is a few lines of
NumPy and matplotlib that you could have written yourself.
Change something, run the cell again, and see what happens.
"""
import numpy as np
import matplotlib.pyplot as plt

# Colors that are easy to tell apart (also for color-blind readers).
COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9"]

_size = 6     # how far the graph paper reaches in each direction
_count = 0    # how many arrows we have drawn so far (picks the next color)


def new_grid(size=6):
    """Start a fresh sheet of graph paper that runs from -size to +size."""
    global _size, _count
    _size = size
    _count = 0
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_aspect("equal")                  # one step right = one step up
    if size <= 12:                          # small paper: a line at every whole number
        ax.set_xticks(range(-size, size + 1))
        ax.set_yticks(range(-size, size + 1))
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="black", linewidth=1)   # the x-axis
    ax.axvline(0, color="black", linewidth=1)   # the y-axis
    return ax


def _paper():
    """Give back the sheet we are drawing on. Make one if there is none."""
    if plt.get_fignums():
        return plt.gca()
    return new_grid(_size)


def _next_color():
    """Pick the next color from the list, starting over when we run out."""
    global _count
    color = COLORS[_count % len(COLORS)]
    _count = _count + 1
    return color


def draw_vector(v, start=(0, 0), color=None, label=None, label_at="tip"):
    """Draw the vector v as an arrow that begins at start.

    The label goes just past the tip, or beside the middle of the arrow
    if you say label_at="middle".
    Gives back the tip of the arrow (start + v), so the next arrow
    can begin where this one ended.
    """
    ax = _paper()
    v = np.array(v, dtype=float)
    start = np.array(start, dtype=float)
    if color is None:
        color = _next_color()
    ax.quiver(start[0], start[1], v[0], v[1],
              angles="xy", scale_units="xy", scale=1,   # draw at true size
              color=color, width=0.008, zorder=3)
    tip = start + v
    if label is not None:
        length = np.linalg.norm(v)
        if length == 0:
            spot = tip + np.array([0.0, _size * 0.07])
        elif label_at == "middle":
            sideways = np.array([-v[1], v[0]]) / length   # v turned a quarter turn
            spot = start + v / 2 + sideways * _size * 0.07
        else:
            spot = tip + v / length * _size * 0.07        # just past the tip
        ax.text(spot[0], spot[1], label, color=color,
                fontsize=13, fontweight="bold", ha="center", va="center")
    return tip


def draw_point(p, label=None, color="black"):
    """Draw a dot at the point p, with an optional name next to it."""
    ax = _paper()
    ax.plot(p[0], p[1], "o", color=color, markersize=8, zorder=4)
    if label is not None:
        ax.text(p[0] + _size * 0.04, p[1] + _size * 0.04, label,
                color=color, fontsize=12)


def draw_path(steps, start=(0, 0), labels=None):
    """Draw a list of vectors tip-to-tail, like a trip with several legs.

    Gives back the place where the trip ends.
    """
    place = np.array(start, dtype=float)
    for i, step in enumerate(steps):
        name = labels[i] if labels is not None else None
        place = draw_vector(step, start=place, label=name, label_at="middle")
    return place


def draw_dots(points, color="gray", size=18):
    """Draw many points at once. points is a list of [x, y] pairs."""
    ax = _paper()
    points = np.array(points, dtype=float)
    ax.scatter(points[:, 0], points[:, 1], s=size, color=color, zorder=2)


def draw_skewed_grid(b1, b2, color="#E69F00"):
    """Draw the slanted graph paper you get from the two moves b1 and b2."""
    ax = _paper()
    n = 3 * _size                           # enough lines to cover the sheet
    b1 = np.array(b1, dtype=float)
    b2 = np.array(b2, dtype=float)
    for k in range(-n, n + 1):
        # a line in the b1 direction, shifted over by k copies of b2
        begin = k * b2 - n * b1
        end = k * b2 + n * b1
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)
        # a line in the b2 direction, shifted over by k copies of b1
        begin = k * b1 - n * b2
        end = k * b1 + n * b2
        ax.plot([begin[0], end[0]], [begin[1], end[1]], color=color, linewidth=1, alpha=0.7)


def angle_between(a, b):
    """The angle between vectors a and b, in degrees."""
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    cosine = np.clip(cosine, -1, 1)     # guard against tiny rounding errors
    degrees = np.degrees(np.arccos(cosine))
    return round(float(degrees), 2)     # two decimal places is plenty


def show():
    """Show the picture."""
    plt.show()

In [ ]:
import numpy as np
import importlib, vecdraw
importlib.reload(vecdraw)      # pick up any changes you make to the library
from vecdraw import *

## Warm-up: same skills, new numbers

On paper first, with `a = (1, 2)` and `b = (3, 1)`:

1. What is `2 * a - 1 * b`?
2. Find the weights that reach `(4, 3)`.
3. What will the loop below print?

In [ ]:
a = np.array([1, 2])
b = np.array([3, 1])
print(2 * a - 1 * b)

for c in range(-2, 3):
    print(c, c * a)

## Part 1: Collect every place you can reach

A **list** holds many things in order. `[]` is an empty list, and `.append` adds one more thing to the end.

In [ ]:
places = []
places.append(1 * a)
places.append(2 * a)
places.append(3 * a)
print(places)
print(len(places))     # how many things are in the list

Now collect **every** trip that uses whole minutes from -4 to 4 of each mode.

Before you run this: how many places will be in the list? What will the picture look like? Sketch it.

In [ ]:
places = []
for c in range(-4, 5):
    for d in range(-4, 5):
        places.append(c * a + d * b)
print(len(places))

new_grid(8)
draw_dots(places)
draw_vector(a, label="a")
draw_vector(b, label="b")
show()

## Part 2: Fill in the gaps

The rover does not have to drive whole minutes. Let the computer pick random amounts of time, thousands of times.

`rng.uniform(-8, 8)` picks a random number between -8 and 8.

**The big question:** is there anywhere on this paper the rover cannot reach?

In [ ]:
rng = np.random.default_rng()
print(rng.uniform(-8, 8))
print(rng.uniform(-8, 8))

In [ ]:
places = []
for i in range(6000):
    c = rng.uniform(-8, 8)
    d = rng.uniform(-8, 8)
    places.append(c * a + d * b)

new_grid(8)
draw_dots(places, size=4)
draw_vector(a, label="a")
draw_vector(b, label="b")
show()

## Part 3: The broken rover

Mode b broke. The repair shop replaced it with mode `k = (2, 4)`.

Predict the picture before you run it. Then explain it.

In [ ]:
k = np.array([2, 4])

places = []
for i in range(3000):
    c = rng.uniform(-4, 4)
    d = rng.uniform(-4, 4)
    places.append(c * a + d * k)

new_grid(8)
draw_dots(places, size=4)
draw_vector(k, label="k")
draw_vector(a, label="a")
show()

**Talk about it.**

- What is special about `k` compared with `a`?
- Can the broken rover reach `(3, 6)`? What about `(3, 5)`?
- Could two broken modes ever reach everywhere?

The cell below checks whether a spot is on the broken rover's line. Read it. Why does it compare `y` with `2 * x`?

In [ ]:
spot = np.array([3, 6])
x = spot[0]
y = spot[1]
if y == 2 * x:
    print(spot, "is reachable")
else:
    print(spot, "is NOT reachable")

## Part 4: One move, or none

What can a rover with only **one** mode reach? What about a rover whose only mode is `(0, 0)`? Predict both pictures, then change the cell to test.

In [ ]:
only = np.array([2, -1])

places = []
for i in range(500):
    c = rng.uniform(-4, 4)
    places.append(c * only)

new_grid(8)
draw_dots(places, size=4)
draw_vector(only, label="only")
show()

## The name for this

The set of **all** places you can reach with your moves is called the **span** of those moves.

- The span of `a` and `b` is the whole plane.
- The span of `a` and `k` is just a line.

## Your turn: name that span

For each pair, predict the span on paper: a point, a line, or the whole plane. Then test with the cell below.

1. `(2, 1)` and `(-4, -2)`
2. `(1, 0)` and `(0, 1)`
3. `(1, 1)` and `(1, -1)`
4. `(0, 0)` and `(1, 3)`
5. `(3, 6)` and `(1, 2)`

In [ ]:
first = np.array([2, 1])       # change me
second = np.array([-4, -2])    # change me

places = []
for i in range(6000):
    c = rng.uniform(-8, 8)
    d = rng.uniform(-8, 8)
    places.append(c * first + d * second)

new_grid(8)
draw_dots(places, size=4)
show()

## Check yourself

On paper.

1. With moves `(1, 2)` and `(2, 4)`, can you reach `(5, 10)`? Can you reach `(5, 9)`?
2. Describe the span of `(1, 3)` in words.
3. True or false: the span of two moves always includes `(0, 0)`. Why?
4. Give two moves whose span is the whole plane, and two whose span is a line.

---
# The Deep End

## Deep 1: Fenced-in weights

What do you reach if the weights have to follow a rule? Predict each shape, then change the cell.

1. Both weights between 0 and 1.
2. Both weights 0 or more (no reverse gear).
3. The two weights add up to exactly 1. (Hint: pick `c`, then set `d = 1 - c`.)

How big is shape number 1? Count squares on graph paper.

*This leads to: the determinant, which measures that area (Module 10).*

In [ ]:
places = []
for i in range(3000):
    c = rng.uniform(0, 1)      # change the fences
    d = rng.uniform(0, 1)
    places.append(c * a + d * b)

new_grid(8)
draw_dots(places, size=4)
draw_vector(a, label="a")
draw_vector(b, label="b")
show()

## Deep 2: Whole minutes only

Go back to whole-number weights. The dots from `a = (1, 2)` and `b = (3, 1)` skip most of the grid points.

What **fraction** of grid points do they land on? The cell counts for you. Try these pairs and look for a rule that predicts the answer from the four numbers:

- `(1, 0)` and `(0, 1)`
- `(2, 0)` and `(0, 2)`
- `(2, 1)` and `(1, 3)`
- `(3, 0)` and `(1, 2)`

*This leads to: the determinant again (Module 10).*

In [ ]:
first = np.array([1, 2])
second = np.array([3, 1])

landed = set()                      # a set keeps each spot only once
for c in range(-40, 41):
    for d in range(-40, 41):
        spot = c * first + d * second
        if 0 <= spot[0] < 30 and 0 <= spot[1] < 30:
            landed.add((int(spot[0]), int(spot[1])))

grid_points = 30 * 30               # whole-number points from 0 to 29, both ways
print(len(landed), "out of", grid_points)
print("1 out of every", grid_points / len(landed))

## Deep 3: Three numbers

Vectors can have three numbers: east, north and **up**. Take the moves `p = (1, 0, 1)` and `q = (0, 1, 1)`.

- Can you reach `(2, 3, 5)`? Can you reach `(2, 3, 4)`? Work it out on paper.
- What shape is the span? Run the cell, then drag the idea around in your head: is it a line, a flat sheet, or all of space?

This cell uses matplotlib directly, with no helper library.

In [ ]:
import matplotlib.pyplot as plt

p = np.array([1, 0, 1])
q = np.array([0, 1, 1])

places = []
for i in range(2000):
    c = rng.uniform(-3, 3)
    d = rng.uniform(-3, 3)
    places.append(c * p + d * q)
places = np.array(places)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(places[:, 0], places[:, 1], places[:, 2], s=3)
ax.set_xlabel("east")
ax.set_ylabel("north")
ax.set_zlabel("up")
ax.view_init(elev=15, azim=-25)     # change these two numbers to look from another side
plt.show()

## Deep 4: The span of red and green

Every color in this picture is `c * red + d * green` with weights from 0 to 1. Which colors are missing, and why?

Change it to show the span of red and blue. Then try `red` and `white = (1, 1, 1)`.

In [ ]:
red = np.array([1, 0, 0])
green = np.array([0, 1, 0])

picture = np.zeros((50, 50, 3))          # 50 rows, 50 columns, 3 numbers per pixel
for row in range(50):
    for col in range(50):
        c = row / 49
        d = col / 49
        picture[row, col] = np.clip(c * red + d * green, 0, 1)

plt.figure(figsize=(4, 4))
plt.imshow(picture)
plt.xlabel("more green")
plt.ylabel("more red")
plt.show()

## Deep 5: The number-line cousin

Forget arrows for a minute. Using **whole-number** weights `x` and `y` (negatives allowed), which numbers can `4 * x + 6 * y` make? Which numbers can `3 * x + 5 * y` make?

Find the rule. Then test it on `6 * x + 15 * y`.

In [ ]:
reachable = set()
for x in range(-20, 21):
    for y in range(-20, 21):
        number = 4 * x + 6 * y
        if 0 <= number <= 30:
            reachable.add(number)
print(sorted(reachable))